## Point 3 

NER and keyphrase extraction:
- Run a domain-adapted NER pipeline (spaCy/transformer) + keyphrase extraction (YAKE/KeyBERT) over text chunks.
- Map mentions to ontology classes (entity typing).
- Resolve coreference (merge aliases, e.g., “vitamin C” ↔ “ascorbic acid”).
- Deliverable: entities.jsonl with canonical IDs and mention spans.

Deliverable contains one JSON object per canonical entity: each entity contains a stable ID, a canonical label, type (one of your ontology classes), synonyms/aliases, and all mention occurrences with text spans + provenance (page, section, char offsets).

Inputs are:
- raw_text.jsonl and tables/*.csv
- ontology.yaml



**Provided packages to perform NER**

**spaCy** = fast, production-ready NLP framework with solid pretrained NER + rule-based matchers (perfect for domain patterns) and easy pipelines.

**Hugging Face Transformers** = access to SOTA token classifiers; great for recall and easy to swap models; also gives you zero-shot classification for typing.

**Flair** = lightweight sequence tagger that often picks up entities missed by others; complements spaCy/HF in an ensemble.

Use them together: spaCy for structure + rules, HF for strong neural NER (and zero-shot typing), Flair as a catcher’s mitt to boost recall. Then reconcile outputs, map to ontology classes, and export entities.jsonl.


In [1]:
import json, re, unicodedata
from collections import defaultdict, Counter
from pathlib import Path

RAW = Path("data/raw_text.jsonl")

def norm(s: str) -> str:
    # Unicode fold + lowercase + collapse whitespace; keep hyphens so we can match variants
    s = unicodedata.normalize("NFKD", s).casefold().strip()
    s = re.sub(r"\s+", " ", s)
    return s

# ---------- Helpers to build robust patterns ----------
def esc_variants(term: str) -> str:
    """
    Build a regex that matches normalized text variants:
    - spaces -> [\\s-]+ (space or hyphen)
    - keep digits, allow hyphen/space around digits in common constructs (omega-3, type 2)
    Assumes we'll search in casefolded, normalized text.
    """
    t = term.strip().casefold()
    # protect regex metachars by escaping first
    t = re.escape(t)
    # allow space/hyphen variants
    t = t.replace(r"\ ", r"[\s\-]+")
    # special: omega-3 / omega 3; type 2 diabetes / type-2-diabetes already covered by space->class
    return t

def compile_list(terms, word_boundaries=True):
    alts = [esc_variants(t) for t in terms]
    if not alts:
        return None
    alt = "(?:" + "|".join(sorted(set(alts), key=len, reverse=True)) + ")"
    if word_boundaries:
        return re.compile(rf"\b{alt}\b", re.I)
    return re.compile(alt, re.I)

# --- Existing domain patterns ---
vitamin_rx = re.compile(r"\bvitamin[s]?\s+[a-z](?:\d+)?\b", re.I)  # search on normalized text
minerals_list = ["calcium","iron","zinc","iodine","selenium","potassium","magnesium","phosphorus"]
macro_micro_list = [
    "fibre","fiber","saturated fat","unsaturated fat","omega-3","omega-6",
    "protein","carbohydrate","sodium","salt","sugars","free sugar","added sugar"
]
food_groups = [
    "cereals","grains","whole grain","vegetables","berries","fruits","legumes","pulses",
    "nuts","seeds","fish","seafood","red meat","processed meat","poultry","milk","dairy",
    "eggs","fats","oils","beverages","alcohol","potatoes","rye","oats","barley"
]
tech_terms = [
    "smoking","salting","nitrite","curing","fermentation","pasteurisation","pasteurization",
    "boiling","frying","baking","grilling","fortification","iodised salt","iodized salt"
]
guideline_verbs = ["limit","reduce","increase","prefer","choose","eat more","aim to","replace","avoid"]

# NEW: healthOutcome + environmentImpact patterns
health_outcome_terms = [
    "cardiovascular disease","ischemic heart disease","stroke","type 2 diabetes","obesity","overweight",
    "hypertension","high blood pressure","blood pressure","cholesterol","ldl cholesterol","hdl cholesterol",
    "triglycerides","cancer","colorectal cancer","all-cause mortality","mortality","morbidity","inflammation",
    "insulin resistance","metabolic syndrome"
]
environment_terms = [
    "greenhouse gas emissions","ghg emissions","carbon footprint","climate impact","environmental impact",
    "land use","water use","water footprint","nitrogen footprint","phosphorus footprint","biodiversity",
    "biodiversity loss","ecological footprint","emissions","food system emissions","sustainability"
]

risk_intro = r"(?:risk|risk of|associated with|linked to|higher odds of|increases|reduces)"

# Precompiled alternations against normalized text
minerals_rx   = compile_list(minerals_list)
macromicro_rx = compile_list(macro_micro_list)
foods_rx      = compile_list(food_groups)
tech_rx       = compile_list(tech_terms)
health_rx     = compile_list(health_outcome_terms)
env_rx        = compile_list(environment_terms)

# heads we accept after risk-intros (guard against vague tails)
risk_heads = compile_list([
    "cardiovascular disease","ischemic heart disease","type 2 diabetes","obesity","overweight",
    "hypertension","high blood pressure","blood pressure","ldl cholesterol","hdl cholesterol",
    "triglycerides","cancer","colorectal cancer","mortality","all-cause mortality","stroke",
    "cholesterol","inflammation"
])

deny_tails = re.compile(r"\b(of|to|in|for|on|with|by|at|from)$")

def per_chunk_increment(counter: Counter, key: str, seen: set):
    if key not in seen:
        counter[key] += 1
        seen.add(key)

gaz = defaultdict(Counter)
provenance = defaultdict(lambda: defaultdict(list))  # provenance[category][term] -> list of {"chunk","section","page","excerpt"}

def add_prov(cat, term, chunk_id, section, page, text_norm, match_span, max_len=160):
    start, end = match_span
    s = max(0, start - 60)
    e = min(len(text_norm), end + 60)
    snippet = text_norm[s:e]
    if len(provenance[cat][term]) < 3:  # cap examples to keep file small
        provenance[cat][term].append({
            "chunk": chunk_id, "section": section, "page": page,
            "excerpt": snippet
        })

with open(RAW, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        section = obj.get("section") or ""
        text = obj.get("text","")
        page = obj.get("page")
        cid = obj.get("id") or obj.get("chunk_id") or f"chunk_{page}"

        # Skip likely non-content sections (optional)
        section_raw = (obj.get("section") or obj.get("heading") or "").lower()
        if any(s in section_raw for s in ["table of contents","contents","acknowledgement","acknowledgment","reference","bibliography"]):
            continue

        section_n = norm(section)
        text_n = norm(text)

        # Per-chunk seen sets for doc-frequency counting
        seen_nutr, seen_ing, seen_tech, seen_guid, seen_health, seen_env = [set() for _ in range(6)]

        # ------- Nutrients -------
        for m in vitamin_rx.finditer(text_n):
            per_chunk_increment(gaz["nutrient"], m.group(0), seen_nutr)
            add_prov("nutrient", m.group(0), cid, section_n, page, text_n, m.span())

        for rx in [minerals_rx, macromicro_rx]:
            if not rx: continue
            for m in rx.finditer(text_n):
                term = m.group(0)
                per_chunk_increment(gaz["nutrient"], term, seen_nutr)
                add_prov("nutrient", term, cid, section_n, page, text_n, m.span())

        # ------- Ingredients -------
        if foods_rx:
            for m in foods_rx.finditer(text_n):
                term = m.group(0)
                per_chunk_increment(gaz["ingredient"], term, seen_ing)
                add_prov("ingredient", term, cid, section_n, page, text_n, m.span())

        # ------- Techniques -------
        if tech_rx:
            for m in tech_rx.finditer(text_n):
                term = m.group(0)
                per_chunk_increment(gaz["technique"], term, seen_tech)
                add_prov("technique", term, cid, section_n, page, text_n, m.span())

        # ------- Guidelines (simple verb + noun-phrase) -------
        for v in guideline_verbs:
            # require boundary after verb; capture up to 6 tokens tail
            for m in re.finditer(rf"\b{re.escape(v)}\s+[a-z][a-z\- ]{{2,60}}", text_n, re.I):
                phrase = m.group(0).strip()
                phrase = re.sub(r"\s+", " ", phrase)
                if not deny_tails.search(phrase.split()[-1]):
                    per_chunk_increment(gaz["guideline"], phrase, seen_guid)
                    add_prov("guideline", phrase, cid, section_n, page, text_n, m.span())

        # ------- Health Outcomes -------
        if health_rx:
            for m in health_rx.finditer(text_n):
                term = m.group(0)
                per_chunk_increment(gaz["healthOutcome"], term, seen_health)
                add_prov("healthOutcome", term, cid, section_n, page, text_n, m.span())

        # Risk-intro patterns: “… reduces blood pressure”, “… increases LDL cholesterol”
        risk_pat = re.compile(rf"\b{risk_intro}\s+([a-z][a-z\- ]{{2,60}}?)\b", re.I)
        for m in risk_pat.finditer(text_n):
            tail = m.group(1).strip()
            tail = re.sub(r"\s+", " ", tail)
            if deny_tails.search(tail.split()[-1]):
                continue
            # Accept only if tail contains one of our outcome heads (precompiled)
            if risk_heads and risk_heads.search(tail):
                per_chunk_increment(gaz["healthOutcome"], tail, seen_health)
                add_prov("healthOutcome", tail, cid, section_n, page, text_n, m.span(1))

        # ------- Environmental Impacts -------
        if env_rx:
            for m in env_rx.finditer(text_n):
                term = m.group(0)
                per_chunk_increment(gaz["environmentImpact"], term, seen_env)
                add_prov("environmentImpact", term, cid, section_n, page, text_n, m.span())

        # common abbrev
        if re.search(r"\bghg\b", text_n):
            per_chunk_increment(gaz["environmentImpact"], "ghg emissions", seen_env)
            # provenance without exact span: approximate
            i = text_n.find("ghg")
            add_prov("environmentImpact", "ghg emissions", cid, section_n, page, text_n, (i, i+3))

# Keep items seen >=2 chunks (document frequency)
THRESHOLD = 2
seed_vocab = {k: sorted([term for term, c in cnts.items() if c >= THRESHOLD]) for k, cnts in gaz.items()}

print("Seeds summary (doc-frequency):", {k: len(v) for k, v in seed_vocab.items()})
for k, terms in seed_vocab.items():
    print(f"\n{k.upper()} ({len(terms)}):")
    for t in terms[:60]:
        print(" •", t)

# Write outputs
with open("data/seed_vocabularies.json", "w", encoding="utf-8") as out:
    json.dump(seed_vocab, out, ensure_ascii=False, indent=2)

with open("data/seed_provenance.json", "w", encoding="utf-8") as out:
    json.dump(provenance, out, ensure_ascii=False, indent=2)

print("\n✅ Saved seed_vocabularies.json (doc-frequency) and seed_provenance.json (examples)")


Seeds summary (doc-frequency): {'nutrient': 26, 'ingredient': 25, 'guideline': 2, 'environmentImpact': 10, 'technique': 6, 'healthOutcome': 15}

NUTRIENT (26):
 • added sugar
 • calcium
 • carbohydrate
 • fiber
 • fibre
 • free sugar
 • iodine
 • iron
 • magnesium
 • phosphorus
 • potassium
 • protein
 • salt
 • saturated fat
 • selenium
 • sugars
 • unsaturated fat
 • vitamin a
 • vitamin b12
 • vitamin b6
 • vitamin c
 • vitamin d
 • vitamin d3
 • vitamin e
 • vitamin k
 • zinc

INGREDIENT (25):
 • alcohol
 • barley
 • berries
 • beverages
 • cereals
 • dairy
 • eggs
 • fats
 • fish
 • fruits
 • grains
 • legumes
 • milk
 • nuts
 • oats
 • oils
 • potatoes
 • poultry
 • processed meat
 • red meat
 • rye
 • seafood
 • seeds
 • vegetables
 • whole grain

GUIDELINE (2):
 • aim to influence the population
 • increase blood cholesterol levels

ENVIRONMENTIMPACT (10):
 • biodiversity
 • biodiversity loss
 • carbon footprint
 • climate impact
 • emissions
 • environmental impact
 • greenhou

In [3]:
%pip uninstall -y camelot-py torchvision
%pip install -U spacy
%pip check


Found existing installation: camelot-py 1.0.9
Uninstalling camelot-py-1.0.9:
  Successfully uninstalled camelot-py-1.0.9
Found existing installation: torchvision 0.17.2
Uninstalling torchvision-0.17.2:
  Successfully uninstalled torchvision-0.17.2
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
torchaudio 2.2.2 requires torch, which is not installed.
Note: you may need to restart the kernel to use updated packages.


In [1]:
%pip install -q spacy

Note: you may need to restart the kernel to use updated packages.


In [6]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Step 3 (improved, notebook-friendly):
- Gazetteer NER via spaCy + PhraseMatcher
- Overlap resolution + per-chunk dedup
- Alias/coref merge with union-find + protected pairs
- Stable IDs using ontology prefix
- Writes entities.jsonl
"""

import json, re, unicodedata, hashlib
from pathlib import Path
from collections import defaultdict, Counter

import yaml
import spacy
from spacy.matcher import PhraseMatcher
from spacy.util import filter_spans
from spacy.tokens import Span

# ──────────────────────────────────────────────────────────────────────────────
# Paths (same structure you used)
# ──────────────────────────────────────────────────────────────────────────────
RAW_PATH   = Path("data/raw_text.jsonl")
SEEDS_PATH = Path("data/new_seed.json")
ONTO_PATH  = Path("data/ontology.yaml")
OUT_PATH   = Path("data/entities_new.jsonl")

# ──────────────────────────────────────────────────────────────────────────────
# Ontology essentials (more robust base_uri/prefix handling)
# ──────────────────────────────────────────────────────────────────────────────
onto = yaml.safe_load(ONTO_PATH.read_text(encoding="utf-8")) or {}
BASE_URI = (
    onto.get("meta", {}).get("base_uri")
    or onto.get("base_uri")
    or "http://example.org/food#"
)
PREFIX = onto.get("prefixes", {}).get("ex", BASE_URI)

print("Base URI:", BASE_URI)
print("Prefix  :", PREFIX)

# Allowed classes (extend if ontology has others)
VALID_CLASSES = {
    "nutrient": "nutrient",
    "ingredient": "ingredient",
    "technique": "technique",
    "dietaryGuideline": "dietaryGuideline",
    "healthOutcome": "healthOutcome",
    "environmentImpact": "environmentImpact",
}

# Map seed groups -> ontology classes (keys must match your seed groups)
LABEL_MAP = {
    "nutrient": "nutrient",
    "ingredient": "ingredient",
    "technique": "technique",
    "guideline": "dietaryGuideline",
    "healthOutcome": "healthOutcome",
    "environmentImpact": "environmentImpact",
}

# ──────────────────────────────────────────────────────────────────────────────
# Load seeds
# ──────────────────────────────────────────────────────────────────────────────
seeds = json.loads(SEEDS_PATH.read_text(encoding="utf-8"))
for k in list(seeds.keys()):
    if k not in LABEL_MAP:
        print("Skipping unknown seed group:", k)

print("Loaded seeds:", {k: len(v) for k, v in seeds.items() if k in LABEL_MAP})

# ──────────────────────────────────────────────────────────────────────────────
# Normalization helpers
# ──────────────────────────────────────────────────────────────────────────────
def norm_text(s: str) -> str:
    s = unicodedata.normalize("NFKD", s).casefold().strip()
    s = re.sub(r"\s+", " ", s)
    return s

def alias_norm(s: str) -> str:
    # Aggressive normalization for alias matching
    s = unicodedata.normalize("NFKD", s).casefold()
    s = s.replace("-", " ")
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def safe_slug(s: str) -> str:
    base = alias_norm(s)
    base = re.sub(r"[^a-z0-9]+", "_", base).strip("_")
    if not base:
        base = hashlib.md5(s.encode("utf-8")).hexdigest()[:8]
    return base

def context_window_sentence(doc, start_char):
    # Sentence text containing the span; fallback to None
    for sent in doc.sents:
        if sent.start_char <= start_char < sent.end_char:
            return sent.text
    return None

# ──────────────────────────────────────────────────────────────────────────────
# spaCy pipeline: blank English + sentencizer + PhraseMatcher (fast gazetteer)
# ──────────────────────────────────────────────────────────────────────────────
nlp = spacy.blank("en")
if "sentencizer" not in nlp.pipe_names:
    nlp.add_pipe("sentencizer")

matcher = PhraseMatcher(nlp.vocab, attr="LOWER")

def add_with_variants(label: str, term: str):
    term = term.strip()
    if not term:
        return
    docs = [nlp.make_doc(term)]
    # add common hyphen/space variants
    if "-" in term:
        docs.append(nlp.make_doc(term.replace("-", " ")))
    if " " in term:
        docs.append(nlp.make_doc(term.replace(" ", "-")))
    # de-dup variants
    _seen = set()
    uniq_docs = []
    for d in docs:
        key = d.text.lower()
        if key not in _seen:
            _seen.add(key)
            uniq_docs.append(d)
    matcher.add(label, uniq_docs)

added = 0
for group, terms in seeds.items():
    label = LABEL_MAP.get(group)
    if not label:
        continue
    for term in terms:
        add_with_variants(label, term)
        added += 1

print(f"PhraseMatcher loaded with {added} seed terms across {len([k for k in seeds if k in LABEL_MAP])} groups.")

# ──────────────────────────────────────────────────────────────────────────────
# Section filtering (skip non-content sections)
# ──────────────────────────────────────────────────────────────────────────────
NONCONTENT = (
    "table of contents", "contents", "acknowledgement", "acknowledgment",
    "reference", "references", "bibliography", "appendix"
)

# ──────────────────────────────────────────────────────────────────────────────
# Collect mentions (gazetteer matches) with overlap resolution & per-chunk dedup
# ──────────────────────────────────────────────────────────────────────────────
mentions = []

with RAW_PATH.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        obj = json.loads(line)
        text = obj.get("text", "")
        section = obj.get("section", "") or obj.get("heading", "") or ""
        if any(s in section.lower() for s in NONCONTENT):
            continue

        doc = nlp(text)

        # Gather spans from matcher
        spans = []
        for match_id, start, end in matcher(doc):
            label = nlp.vocab.strings[match_id]  # already ontology class name
            # Only accept labels defined in VALID_CLASSES (normalized compare)
            if label not in VALID_CLASSES.values():
                continue
            spans.append(Span(doc, start, end, label=label))

        # Resolve overlaps (keep best/longest)
        spans = filter_spans(spans)

        # Per-chunk dedup: identical (type, start, end) once
        seen = set()
        for sp in spans:
            key = (sp.label_, sp.start_char, sp.end_char)
            if key in seen:
                continue
            seen.add(key)

            mentions.append({
                "chunk_id": obj.get("id") or obj.get("chunk_id") or f"raw:{i:06d}",
                "page": obj.get("page"),
                "section": section,
                "type": sp.label_,
                "text_span": [int(sp.start_char), int(sp.end_char)],
                "surface": sp.text,
                "context": context_window_sentence(doc, sp.start_char),
            })

print(f"Collected {len(mentions)} mentions.")
for x in mentions[:5]:
    print(x)

# ──────────────────────────────────────────────────────────────────────────────
# Alias/coref merge (union-find) + protected pairs
# ──────────────────────────────────────────────────────────────────────────────
ALIASES = {
    "ascorbic acid": ["vitamin c"],
    "vitamin c": ["ascorbic acid"],
    "folate": ["folic acid","vitamin b9"],
    "cobalamin": ["vitamin b12"],
    "retinol": ["vitamin a"],
    "omega-3": ["n-3","omega 3","n3"],
    "omega-6": ["n-6","omega 6","n6"],
    "polyunsaturated fatty acids": ["pufa","polyunsaturated fat","polyunsaturated fats"],
    "monounsaturated fatty acids": ["mufa","monounsaturated fat","monounsaturated fats"],
    "saturated fatty acids": ["sfa","saturated fat","saturated fats"],
}

DO_NOT_MERGE = {
    tuple(sorted(("added sugar","free sugar"))),
    tuple(sorted(("red meat","processed meat"))),
    tuple(sorted(("ldl cholesterol","hdl cholesterol"))),
    tuple(sorted(("land use","water use"))),
}

class UnionFind:
    def __init__(self):
        self.parent = {}
    def find(self, x):
        self.parent.setdefault(x, x)
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]
    def union(self, a, b):
        pa, pb = self.find(a), self.find(b)
        if pa != pb:
            self.parent[pb] = pa

def build_alias_unionfind():
    uf = UnionFind()
    for k, vs in ALIASES.items():
        ak = alias_norm(k)
        for v in vs:
            uf.union(ak, alias_norm(v))
    return uf

uf = build_alias_unionfind()

# ──────────────────────────────────────────────────────────────────────────────
# Merge mentions into canonical entities (per type), emit stable IDs
# ──────────────────────────────────────────────────────────────────────────────
by_type = defaultdict(list)
for m in mentions:
    by_type[m["type"]].append(m)

entities = {}

def make_id(label, etype):
    # ID includes type prefix for collision safety
    return f"{PREFIX}{etype}_{safe_slug(label)}"

for etype, mlist in by_type.items():
    # bucket by alias representative
    buckets = defaultdict(list)
    for m in mlist:
        key = alias_norm(m["surface"])
        rep = uf.find(key)
        # protect pairs that should not merge
        if tuple(sorted((key, rep))) in DO_NOT_MERGE:
            rep = key
        buckets[(etype, rep)].append(m)

    # build canonical entity per bucket
    for (etype, rep), ms in buckets.items():
        # choose most frequent surface form (case-preserved) as label
        cap_counts = Counter([m["surface"] for m in ms])
        label = max(cap_counts, key=cap_counts.get)
        eid = make_id(label, etype)

        if eid not in entities:
            entities[eid] = {
                "id": eid,
                "type": etype,
                "label": label,
                "aliases": sorted({alias_norm(m["surface"]) for m in ms
                                   if alias_norm(m["surface"]) != alias_norm(label)}),
                "source": "sustainable-health-from-food.pdf",
                "mentions": [],
            }
        entities[eid]["mentions"].extend(ms)

# finalize: order mentions and write JSONL
for e in entities.values():
    e["mentions"].sort(key=lambda m: ((m.get("page") or 0), m["text_span"][0]))

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUT_PATH.open("w", encoding="utf-8") as out:
    for e in entities.values():
        out.write(json.dumps(e, ensure_ascii=False) + "\n")

print(f"✅ Wrote {len(entities)} canonical entities -> {OUT_PATH}")


Base URI: http://example.org/food#
Prefix  : http://example.org/food#
Loaded seeds: {'nutrient': 35, 'ingredient': 43, 'guideline': 4, 'environmentImpact': 13, 'technique': 11, 'healthOutcome': 22}
PhraseMatcher loaded with 128 seed terms across 6 groups.
Collected 1537 mentions.
{'chunk_id': 'raw:000001', 'page': 5, 'section': 'Preface', 'type': 'ingredient', 'text_span': [556, 560], 'surface': 'salt', 'context': 'Finnish food habits have improved in recent decades, but the challenges that remain are excessive salt and saturated fat intakes and insufficient fibre intake.'}
{'chunk_id': 'raw:000001', 'page': 5, 'section': 'Preface', 'type': 'nutrient', 'text_span': [575, 578], 'surface': 'fat', 'context': 'Finnish food habits have improved in recent decades, but the challenges that remain are excessive salt and saturated fat intakes and insufficient fibre intake.'}
{'chunk_id': 'raw:000001', 'page': 5, 'section': 'Preface', 'type': 'dietaryGuideline', 'text_span': [651, 659], 'surface'

In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
YAKE-based keyphrase extraction for enrichment.
Run this AFTER you’ve generated data/entities.jsonl.
It finds salient multi-word phrases to expand your seed vocabulary.
"""

import json, re, unicodedata
from pathlib import Path
from collections import Counter
import yake

# ──────────────────────────────────────────────────────────────────────────────
# Paths
# ──────────────────────────────────────────────────────────────────────────────
RAW_PATH = Path("data/raw_text.jsonl")
ENTITIES_PATH = Path("data/entities_new.jsonl")
OUT_PATH = Path("data/yake_candidates.json")

# ──────────────────────────────────────────────────────────────────────────────
# Helper: normalization
# ──────────────────────────────────────────────────────────────────────────────
def norm(s: str) -> str:
    s = unicodedata.normalize("NFKD", s).casefold().strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^\w\s]", "", s)
    return s

# Load canonical entities & aliases from previous step
existing_terms = set()
if ENTITIES_PATH.exists():
    with ENTITIES_PATH.open(encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            existing_terms.add(norm(obj["label"]))
            for a in obj.get("aliases", []):
                existing_terms.add(norm(a))
print(f"Loaded {len(existing_terms)} existing entity terms to filter out.")

# ──────────────────────────────────────────────────────────────────────────────
# Configure YAKE
# ──────────────────────────────────────────────────────────────────────────────
# Adjust n (n-gram size) and top_n based on your doc length
kw_extractor = yake.KeywordExtractor(
    lan="en", n=3, dedupLim=0.9, top=15, features=None
)

# ──────────────────────────────────────────────────────────────────────────────
# Run YAKE over text chunks
# ──────────────────────────────────────────────────────────────────────────────
candidates = Counter()

with RAW_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        text = obj.get("text", "")
        if not text.strip():
            continue
        section = obj.get("section", "")
        page = obj.get("page", None)

        try:
            kws = kw_extractor.extract_keywords(text)
        except Exception as e:
            print("YAKE failed on chunk", obj.get("id"), ":", e)
            continue

        for kw, score in kws:
            key = norm(kw)
            # skip if too short/long or already known
            if len(key.split()) < 1 or len(key.split()) > 6:
                continue
            if key in existing_terms:
                continue
            if score > 0.3:  # optional: skip low-salience (YAKE scores 0–1; lower = better)
                continue
            candidates[key] += 1

print(f"Collected {len(candidates)} distinct new phrases before filtering.")

# ──────────────────────────────────────────────────────────────────────────────
# Filter and rank top candidates
# ──────────────────────────────────────────────────────────────────────────────
filtered = [
    {"term": term, "count": count}
    for term, count in candidates.most_common()
    if count >= 2  # only keep those appearing in ≥2 chunks
]

print(f"Retained {len(filtered)} candidate keyphrases after filtering.")

# ──────────────────────────────────────────────────────────────────────────────
# Save results
# ──────────────────────────────────────────────────────────────────────────────
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUT_PATH.open("w", encoding="utf-8") as out:
    json.dump(filtered, out, ensure_ascii=False, indent=2)

print(f"✅ Wrote {len(filtered)} YAKE candidates -> {OUT_PATH}")


Loaded 83 existing entity terms to filter out.
Collected 1110 distinct new phrases before filtering.
Retained 251 candidate keyphrases after filtering.
✅ Wrote 251 YAKE candidates -> data/yake_candidates.json


In [6]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Promote YAKE keyphrases into the entity list (PDF-grounded)

Inputs:
  - data/raw_text.jsonl        (PDF chunks)
  - data/entities.jsonl        (from spaCy gazetteer pipeline)
  - data/yake_candidates.json  (from YAKE; accepts either [{"term": "...", ...}, ...] or ["...", ...])

Output:
  - data/entities.v2.jsonl     (merged entities with YAKE-promoted terms)

Rules:
  - skip candidates already covered by existing labels/aliases (fuzzy)
  - find real spans in raw_text and attach provenance (page, chunk, sentence excerpt)
  - assign a conservative ontology class via lexical cues
  - if candidate is very similar to an existing entity of the same class -> add as alias
  - else create a brand-new entity with stable ID
"""

import json, re, unicodedata, hashlib
from pathlib import Path
from collections import defaultdict, Counter

# ──────────────────────────────────────────────────────────────────────────────
# Paths (same style as your other scripts)
# ──────────────────────────────────────────────────────────────────────────────
RAW_PATH   = Path("data/raw_text.jsonl")
ENT_PATH   = Path("data/entities_new.jsonl")
YAKE_PATH  = Path("data/yake_candidates.json")
ONTO_PATH  = Path("data/ontology.yaml")
OUT_PATH   = Path("data/entities_new_yake.jsonl")

# ──────────────────────────────────────────────────────────────────────────────
# Load ontology (for base URI / prefix)
# ──────────────────────────────────────────────────────────────────────────────
onto = {}
if ONTO_PATH.exists():
    import yaml
    onto = yaml.safe_load(ONTO_PATH.read_text(encoding="utf-8")) or {}
BASE_URI = (
    (onto.get("meta", {}) or {}).get("base_uri")
    or onto.get("base_uri")
    or "http://example.org/food#"
)
PREFIX = (onto.get("prefixes", {}) or {}).get("ex", BASE_URI)
print("Prefix:", PREFIX)

# ──────────────────────────────────────────────────────────────────────────────
# Normalization & helpers
# ──────────────────────────────────────────────────────────────────────────────
def norm_space(s: str) -> str:
    s = unicodedata.normalize("NFKD", s).casefold().strip()
    s = re.sub(r"\s+", " ", s)
    return s

def alias_norm(s: str) -> str:
    s = unicodedata.normalize("NFKD", s).casefold()
    s = s.replace("-", " ")
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def safe_slug(s: str) -> str:
    base = alias_norm(s)
    base = re.sub(r"[^a-z0-9]+","_", base).strip("_")
    if not base:
        base = hashlib.md5(s.encode("utf-8")).hexdigest()[:8]
    return base

def make_id(label: str, etype: str) -> str:
    return f"{PREFIX}{etype}_{safe_slug(label)}"

def alt_variant_regex(term: str) -> re.Pattern:
    """
    liberal regex for a term: allows space<->hyphen swaps and casefolded match
    """
    t = norm_space(term)
    t = re.escape(t).replace(r"\ ", r"[\s\-]+")
    return re.compile(rf"\b{t}\b", re.I)

def sent_window(text: str, start: int, end: int, win: int = 120) -> str:
    a = max(0, start - win)
    b = min(len(text), end + win)
    return text[a:b]

def jaccard(a: str, b: str) -> float:
    sa, sb = set(alias_norm(a).split()), set(alias_norm(b).split())
    if not sa or not sb: return 0.0
    return len(sa & sb) / len(sa | sb)

# ──────────────────────────────────────────────────────────────────────────────
# Load existing entities and build lookups
# ──────────────────────────────────────────────────────────────────────────────
entities = []                 # list of entity dicts
by_id = {}                    # id -> entity
class_to_entities = defaultdict(list)  # type -> [entities]
known_terms = set()           # normalized labels/aliases

if ENT_PATH.exists():
    with ENT_PATH.open(encoding="utf-8") as f:
        for line in f:
            e = json.loads(line)
            entities.append(e)
            by_id[e["id"]] = e
            class_to_entities[e["type"].lower()].append(e)
            known_terms.add(alias_norm(e.get("label","")))
            for a in e.get("aliases", []):
                known_terms.add(alias_norm(a))
print(f"Loaded {len(entities)} existing entities; {len(known_terms)} known terms/aliases.")

# ──────────────────────────────────────────────────────────────────────────────
# Load YAKE candidates (accepts either list of dicts with "term" or list of strings)
# ──────────────────────────────────────────────────────────────────────────────
yake_terms = []
if YAKE_PATH.exists():
    data = json.loads(YAKE_PATH.read_text(encoding="utf-8"))
    if isinstance(data, list):
        for item in data:
            if isinstance(item, dict) and "term" in item:
                yake_terms.append(item["term"])
            elif isinstance(item, str):
                yake_terms.append(item)
    else:
        raise ValueError("yake_candidates.json must be a list of dicts or strings.")
    # dedup and sort
    seen = set()
    uniq = []
    for t in yake_terms:
        n = alias_norm(t)
        if not n or n in seen: 
            continue
        seen.add(n)
        uniq.append(t)
    yake_terms = uniq
else:
    raise FileNotFoundError(f"Missing {YAKE_PATH}")

print(f"YAKE candidates loaded: {len(yake_terms)} terms.")

# ──────────────────────────────────────────────────────────────────────────────
# Conservative lexical cues for typing (no external models)
# ──────────────────────────────────────────────────────────────────────────────
CUE = {
    "nutrient": [
        r"\bvitamin\s+[a-z]\d?\b", r"\b(amino|fatty)\s+acid(s)?\b", r"\bminerals?\b",
        r"\bomega[\s\-]?\d+\b", r"\b(fibre|fiber|protein|carbohydrate|starch|sugar|lipid)s?\b",
        r"\b(ldl|hdl)\s+cholesterol\b", r"\bcholesterol\b", r"\btriglycerides?\b",
        r"\b(iron|calcium|iodine|zinc|selenium|magnesium|potassium|sodium|phosphorus)\b",
        r"\b(ascorbic acid|folate|folic acid|retinol|cobalamin|riboflavin|niacin|thiamine|choline)\b",
    ],
    "technique": [
        r"\b(fermentation|boiling|steaming|frying|baking|grilling|pasteuri[sz]ation|curing|salting|smoking)\b",
        r"\b(dehydration|drying|pickling|canning|blanching|stewing|pressure cooking|sous[\s\-]?vide)\b",
        r"\b(fortification|iodi[sz]ed\s+salt)\b",
    ],
    "ingredient": [
        r"\b(whole\s+grain|grains?|cereals?|rye|oats?|barley|rice|potatoes?)\b",
        r"\b(vegetables?|fruits?|berries|legumes?|pulses|nuts?|seeds?)\b",
        r"\b(fish|seafood|salmon|herring|mackerel|tuna|poultry|eggs?|dairy|milk|yog?h?urt|cheese)\b",
        r"\b(red\s+meat|processed\s+meat)\b",
    ],
    "dietaryGuideline": [
        r"\b(limit|reduce|increase|prefer|choose|replace|avoid|eat\s+more|aim\s+to)\b",
        r"\b(intake|consumption)\b",
    ],
    "healthOutcome": [
        r"\b(cardiovascular disease|ischa?emic heart disease|stroke|type\s*2\s*diabetes|obesity|overweight)\b",
        r"\b(hypertension|high\s+blood\s+pressure|blood\s+pressure)\b",
        r"\b(cholesterol|ldl\s+cholesterol|hdl\s+cholesterol|triglycerides?)\b",
        r"\b(cancer|colorectal\s+cancer|mortality|all[\s\-]?cause\s+mortality|morbidity|inflammation)\b",
        r"\b(insulin\s+resistance|metabolic\s+syndrome)\b",
    ],
    "environmentImpact": [
        r"\b(ghg|greenhouse\s+gas(?:es)?\s+emissions|carbon\s+footprint|emissions?)\b",
        r"\b(environmental\s+impact|climate\s+impact|sustainability)\b",
        r"\b(land\s+use|water\s+use|water\s+footprint|nitrogen\s+footprint|phosphorus\s+footprint|biodiversity(?:\s+loss)?)\b",
        r"\becological\s+footprint\b",
    ],
}
cue_rx = {cls: re.compile("|".join(pats), re.I) for cls, pats in CUE.items()}

def guess_class(term: str) -> str:
    for cls, rx in cue_rx.items():
        if rx.search(term):
            return cls
    return "unknown"

# Try to attach as alias to an existing entity of the same class if very similar
def find_entity_alias_target(term: str, etype: str, threshold: float = 0.82):
    best = None
    best_sim = 0.0
    for e in class_to_entities.get(etype.lower(), []):
        sim = max(jaccard(term, e["label"]), *(jaccard(term, a) for a in e.get("aliases", [])) or [0.0])
        if sim > best_sim:
            best_sim, best = sim, e
    return (best if best_sim >= threshold else None)

# ──────────────────────────────────────────────────────────────────────────────
# Scan raw_text for YAKE terms, create mentions, and integrate
# ──────────────────────────────────────────────────────────────────────────────
# We'll search with a liberal regex (space/hyphen variants) per term
added_entities = 0
added_aliases = 0

# Preload raw chunks in memory for one pass
chunks = []
with RAW_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        chunks.append({
            "id": obj.get("id") or obj.get("chunk_id") or None,
            "page": obj.get("page"),
            "section": obj.get("section") or obj.get("heading") or "",
            "text": obj.get("text","") or ""
        })

for term in yake_terms:
    t_norm = alias_norm(term)
    if t_norm in known_terms:
        continue  # already covered

    # find spans across chunks
    pat = alt_variant_regex(term)
    mentions = []
    for ch in chunks:
        text = ch["text"]
        if not text:
            continue
        for m in pat.finditer(text):
            start, end = m.start(), m.end()
            mentions.append({
                "chunk_id": ch["id"],
                "page": ch["page"],
                "section": ch["section"],
                "text_span": [int(start), int(end)],
                "surface": text[start:end],
                "context": sent_window(text, start, end),
            })
    # require at least 2 chunks to reduce noise
    n_chunks = len({m["chunk_id"] for m in mentions if m["chunk_id"]})
    if n_chunks < 2:
        continue

    # guess class from lexical cues
    etype = guess_class(term)

    if etype != "unknown":
        # try alias-attach to an existing entity of same class
        target = find_entity_alias_target(term, etype)
        if target is not None:
            a_set = set(target.get("aliases", []))
            if t_norm not in {alias_norm(x) for x in a_set} and alias_norm(target["label"]) != t_norm:
                a_set.add(term)
                target["aliases"] = sorted(a_set)
                # enrich with a few mentions (cap to avoid explosion)
                cap = max(0, 10 - len(target.get("mentions", [])))
                if cap > 0:
                    target["mentions"].extend(mentions[:cap])
                added_aliases += 1
            continue

    # else create a brand-new entity
    new_type = etype if etype != "unknown" else "keyphrase"  # fallback class if ontology typing is unclear
    label = term
    eid = make_id(label, new_type)
    e = {
        "id": eid,
        "type": new_type,
        "label": label,
        "aliases": [],
        "source": "sustainable-health-from-food.pdf",
        "mentions": mentions[:20],  # cap mentions to keep file small
    }
    entities.append(e)
    by_id[eid] = e
    class_to_entities[new_type.lower()].append(e)
    known_terms.add(t_norm)
    added_entities += 1

print(f"YAKE integration: {added_aliases} aliases attached; {added_entities} new entities created.")

# ──────────────────────────────────────────────────────────────────────────────
# Sort mentions & write entities.v2.jsonl
# ──────────────────────────────────────────────────────────────────────────────
for e in entities:
    e["mentions"] = sorted(
        e.get("mentions", []),
        key=lambda m: ((m.get("page") or 0), m["text_span"][0] if m.get("text_span") else 0)
    )

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUT_PATH.open("w", encoding="utf-8") as out:
    for e in entities:
        out.write(json.dumps(e, ensure_ascii=False) + "\n")

print(f"✅ Wrote merged entities -> {OUT_PATH}")


Prefix: http://example.org/food#
Loaded 83 existing entities; 83 known terms/aliases.
YAKE candidates loaded: 251 terms.
YAKE integration: 0 aliases attached; 0 new entities created.
✅ Wrote merged entities -> data/entities_new_yake.jsonl


In [7]:
import json

# Path to your entities file
path = "data/entities_new.jsonl"

# Load all entities (one JSON object per line)
entities = []
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            entities.append(json.loads(line))

# Print summary
print(f"Total entities: {len(entities)}\n")

# Sort by type and label
entities_sorted = sorted(entities, key=lambda e: (e["type"], e["label"].lower()))

for e in entities_sorted:
    print(f"{e['type']:<20} | {e['label']}")


Total entities: 109

dietaryGuideline     | avoid
dietaryGuideline     | decrease
dietaryGuideline     | increase
dietaryGuideline     | limit
environmentImpact    | biodiversity loss
environmentImpact    | carbon footprint
environmentImpact    | climate change
environmentImpact    | climate impact
environmentImpact    | eutrophication
environmentImpact    | food waste
environmentImpact    | greenhouse gas emissions
environmentImpact    | land use
environmentImpact    | sustainability
environmentImpact    | sustainable food production
environmentImpact    | water footprint
healthOutcome        | blood pressure
healthOutcome        | cancer
healthOutcome        | cardiovascular disease
healthOutcome        | cholesterol
healthOutcome        | colorectal cancer
healthOutcome        | heart disease
healthOutcome        | hypertension
healthOutcome        | inflammation
healthOutcome        | LDL cholesterol
healthOutcome        | morbidity
healthOutcome        | mortality
healthOutcome   